In [ ]:
# @title 🛡️ 嚴格控價版：檔期管理與警示系統
# @markdown 特點：非活動平台強制鎖定牌價、價格倒掛自動警示、視覺簡潔化(無背景色)。

import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import random
from datetime import date, datetime, timedelta

# --- 1. 系統初始化 ---
if 'df_products' not in globals():
    lines = ['寢具線', '家電線', '收納線', '廚具線', '3C周邊']
    data = []
    for i in range(1, 201):
        line = random.choice(lines)
        price = random.choice([800, 1280, 1580, 2200, 3980, 5000])
        data.append({
            'SKU': f'P{1000+i}',
            'Name': f'【{line}】標準商品-{i:03d}',
            'Line': line,
            'ListPrice': price
        })
    df_products = pd.DataFrame(data)

if 'df_history' not in globals():
    df_history = pd.DataFrame(columns=['SKU', 'Name', 'Platform', 'ListPrice', 'FinalPrice', 'Gap', 'Discount', 'StartDate', 'EndDate', 'Activity', 'Memo'])

ALL_PLATFORMS = ['MOMO', 'PChome', 'Yahoo', '蝦皮', '博客來', '官網']
style = {'description_width': 'initial'}

# --- 2. 介面元件 ---

# 篩選與搜尋
w_line_filter = widgets.Dropdown(options=['請選擇線別'] + list(df_products['Line'].unique()), description='線別:', style=style)
w_sku_selector = widgets.Dropdown(options=[], description='產品:', disabled=True, layout=widgets.Layout(width='60%'))

# 價格設定
w_input_list_price = widgets.FloatText(description='商品牌價 $:', value=1000, style=style)
w_target_platform = widgets.Dropdown(options=ALL_PLATFORMS, description='特價平台:', style=style)
w_discount = widgets.FloatSlider(value=0.9, min=0.5, max=1.2, step=0.01, description='設定折數:', readout_format='.0%')
w_final_price_preview = widgets.FloatText(description='= 活動價 $:', disabled=True, style=style)

# 檔期資訊
w_date_start = widgets.DatePicker(description='開始日期:', value=date.today(), style=style)
w_date_end = widgets.DatePicker(description='結束日期:', value=date.today() + timedelta(days=7), style=style)
w_activity_name = widgets.Text(description='活動名稱:', placeholder='例如: 週末閃購', style=style)
w_memo = widgets.Text(description='備註事項:', placeholder='例如: 需鎖定其他通路 (限50字)', layout=widgets.Layout(width='90%'), style=style)

# 按鈕
btn_simulate = widgets.Button(description=" 1. 執行控價模擬", button_style='primary', icon='gavel') # icon changed to gavel (law)
btn_save = widgets.Button(description=" 2. 確認並儲存", button_style='success', icon='check')
out_sim_result = widgets.Output()
out_warning = widgets.Output() # 專門顯示警告的區域

# 歷史查詢元件
w_search_start = widgets.DatePicker(description='查詢起始:', value=date(date.today().year, 1, 1), style=style)
w_search_end = widgets.DatePicker(description='查詢結束:', value=date(date.today().year, 12, 31), style=style)
w_search_sku = widgets.Text(description='搜尋 SKU:', placeholder='留空則顯示全部', style=style)
btn_search_history = widgets.Button(description="查詢紀錄", icon='search')
out_history_table = widgets.Output()

# --- 3. 邏輯處理 ---

def on_line_change(change):
    if change['new'] == '請選擇線別':
        w_sku_selector.options = []
        w_sku_selector.disabled = True
    else:
        filtered = df_products[df_products['Line'] == change['new']]
        opts = [f"{r.SKU} | {r.Name} (${r.ListPrice})" for r in filtered.itertuples()]
        w_sku_selector.options = opts
        w_sku_selector.disabled = False

def on_sku_change(change):
    if not change['new']: return
    try:
        val = change['new']
        db_price = float(val.split('($')[1].replace(')', ''))
        w_input_list_price.value = db_price
    except:
        pass

def auto_calc_price(*args):
    lp = w_input_list_price.value
    w_final_price_preview.value = lp * w_discount.value

def run_simulation(b):
    with out_sim_result:
        clear_output()
        out_warning.clear_output() # 清空之前的警告
        
        list_price = w_input_list_price.value
        event_price = w_final_price_preview.value
        target_plat = w_target_platform.value
        act_name = w_activity_name.value if w_activity_name.value else "(未填寫)"
        date_range = f"{w_date_start.value} ~ {w_date_end.value}"
        
        if list_price <= 0:
            print("⚠️ 錯誤：牌價無效。")
            return

        # ---------------------------------------------
        # 核心邏輯：價格鎖定與衝突檢測
        # ---------------------------------------------
        data_rows = []
        conflict_detected = False
        
        for p in ALL_PLATFORMS:
            if p == target_plat:
                # 這是活動平台
                final = event_price
                note = f"🔥 {act_name}"
                is_target = True
                status_icon = "👉" # 清楚辨識的指標
            else:
                # 其他平台 -> 強制鎖死在牌價 (List Price)
                final = list_price
                note = "🔒 已鎖定 (牌價)"
                is_target = False
                status_icon = " "

            # 檢測：是否有非活動平台的價格 < 活動價？ (即價格倒掛)
            # 正常情況：非活動(1000) > 活動(900) -> OK
            # 異常情況：非活動(1000) < 活動(1200) -> 警告 (代表你活動價設太高，或者折數設錯變漲價)
            price_gap = final - event_price
            
            if final < event_price:
                conflict_detected = True
                note = "❌ 異常！低於活動價"
                status_icon = "⚠️"

            data_rows.append({
                "狀態": status_icon,
                "平台": p,
                "售價": final,
                "與活動價差": price_gap, # 正數代表比活動價貴(正常)，負數代表比活動價便宜(異常)
                "備註說明": note,
                "_is_target": is_target,
                "_is_conflict": (final < event_price)
            })
            
        df_sim = pd.DataFrame(data_rows)
        
        # ---------------------------------------------
        # 顯示結果 (Clean Style)
        # ---------------------------------------------
        sku_display = w_sku_selector.value.split('|')[0] if w_sku_selector.value else "自訂品項"
        print(f"📦 產品: {sku_display} | 牌價: ${list_price:,.0f}")
        
        # 樣式定義：不使用背景色，改用字體粗細與顏色
        def style_rows(row):
            styles = []
            # 預設樣式
            base_style = 'color: black;'
            
            if row['_is_conflict']:
                # 衝突：紅字 + 粗體
                base_style = 'color: #dc3545; font-weight: bold;'
            elif row['_is_target']:
                # 主打：深藍字 + 粗體 (清楚辨識，不刺眼)
                base_style = 'color: #0d6efd; font-weight: bold; border-bottom: 2px solid #0d6efd;'
            else:
                # 其他：灰色字 (代表被鎖定/不重要)
                base_style = 'color: #6c757d;'
                
            return [base_style] * len(row)

        styler = df_sim.style.apply(style_rows, axis=1).format({
            "售價": "${:,.0f}",
            "與活動價差": "${:+,.0f}"
        })
        
        # 隱藏輔助欄位
        if hasattr(styler, "hide"): 
            styler.hide(axis='columns', subset=['_is_target', '_is_conflict'])
        else: 
            styler.hide_columns(['_is_target', '_is_conflict'])
            
        display(styler)
        
        # ---------------------------------------------
        # 警示系統 (Alert System)
        # ---------------------------------------------
        with out_warning:
            if conflict_detected:
                print("
")
                print("🛑🛑🛑 禁止！價格異常警示 🛑🛑🛑")
                print("偵測到「非活動平台」的價格竟然「低於」您設定的活動價！")
                print("請檢查：")
                print("1. 折數是否設定錯誤？(目前設定導致活動價反而變貴)")
                print("2. 牌價輸入是否正確？")
                # 鎖住儲存按鈕 (模擬概念，實際上Colab按鈕無法即時鎖住，但給予強烈文字警告)
                btn_save.button_style = 'danger' # 變紅色
                btn_save.description = "❌ 價格異常，禁止儲存"
            else:
                print("
✅ 價格結構正常：其餘平台已鎖定且高於活動價。")
                btn_save.button_style = 'success'
                btn_save.description = " 2. 確認並儲存"

def save_campaign(b):
    global df_history
    # 防呆：如果是異常狀態，不執行儲存 (雖然按鈕顯示禁止，但邏輯上也要擋)
    if btn_save.button_style == 'danger':
        with out_warning:
            print("❌ 儲存失敗：請先修正價格異常。")
        return

    with out_sim_result:
        if w_date_start.value > w_date_end.value:
            print("❌ 日期錯誤。")
            return
            
        sku_info = w_sku_selector.value if w_sku_selector.value else "Manual-Input"
        sku = sku_info.split(' | ')[0]
        name = sku_info.split(' | ')[1].split(' ($')[0] if '|' in sku_info else "自訂品項"
        
        new_record = {
            'SKU': sku,
            'Name': name,
            'Platform': w_target_platform.value,
            'ListPrice': w_input_list_price.value,
            'FinalPrice': w_final_price_preview.value,
            'Gap': w_input_list_price.value - w_final_price_preview.value,
            'Discount': f"{w_discount.value*100:.0f}%",
            'StartDate': pd.to_datetime(w_date_start.value),
            'EndDate': pd.to_datetime(w_date_end.value),
            'Activity': w_activity_name.value,
            'Memo': w_memo.value
        }
        
        df_history = pd.concat([df_history, pd.DataFrame([new_record])], ignore_index=True)
        print(f"✅ 已儲存紀錄：{sku} - {w_activity_name.value}")

def search_history(b):
    with out_history_table:
        clear_output()
        if df_history.empty:
            print("尚無歷史資料。")
            return
            
        s_date = pd.to_datetime(w_search_start.value)
        e_date = pd.to_datetime(w_search_end.value)
        mask_date = (df_history['StartDate'] <= e_date) & (df_history['EndDate'] >= s_date)
        res = df_history[mask_date]
        
        keyword = w_search_sku.value.strip().upper()
        if keyword:
            res = res[res['SKU'].str.contains(keyword, na=False)]
            
        if res.empty:
            print("🔍 查無符合條件的紀錄。")
        else:
            print(f"📊 查詢結果 ({len(res)}筆):")
            display_df = res.copy()
            display_df['日期區間'] = display_df['StartDate'].dt.strftime('%Y/%m/%d') + "~" + display_df['EndDate'].dt.strftime('%m/%d')
            cols = ['SKU', 'Name', 'Platform', 'FinalPrice', 'Discount', '日期區間', 'Activity']
            
            # 歷史表格保持簡單清晰
            display(display_df[cols].style.format({'FinalPrice': '${:,.0f}'}))

# 綁定
w_line_filter.observe(on_line_change, names='value')
w_sku_selector.observe(on_sku_change, names='value')
w_sku_selector.observe(auto_calc_price, names='value')
w_input_list_price.observe(auto_calc_price, names='value')
w_discount.observe(auto_calc_price, names='value')

btn_simulate.on_click(run_simulation)
btn_save.on_click(save_campaign)
btn_search_history.on_click(search_history)

# --- 4. 畫面佈局 ---
ui_tab1 = widgets.VBox([
    widgets.HBox([w_line_filter, w_sku_selector]),
    widgets.HTML("<hr>"),
    widgets.HTML("<b>💲 價格模擬 (非活動平台將強制鎖定):</b>"),
    widgets.HBox([w_input_list_price, w_discount, w_final_price_preview]),
    widgets.HBox([w_target_platform, widgets.Label(" ")]),
    widgets.HTML("<br><b>📅 檔期設定:</b>"),
    widgets.HBox([w_date_start, w_date_end]),
    widgets.HBox([w_activity_name, w_memo]),
    widgets.HBox([btn_simulate, btn_save]),
    out_warning,
    out_sim_result
])

ui_tab2 = widgets.VBox([
    widgets.HTML("<b>📆 歷史查詢:</b>"),
    widgets.HBox([w_search_start, w_search_end]),
    widgets.HBox([w_search_sku, btn_search_history]),
    widgets.HTML("<hr>"),
    out_history_table
])

tabs = widgets.Tab(children=[ui_tab1, ui_tab2])
tabs.set_title(0, '🛡️ 嚴格控價排程')
tabs.set_title(1, '🗂️ 歷史檔期查詢')

display(tabs)
